# ⚡ AI B-Roll Autopilot — Free GPU Backend

Run Whisper transcription + B-roll matching on a **free Colab T4/A100 GPU**, then connect your local (or hosted) web studio to it.

---

### Before you start
1. `Runtime` → `Change runtime type` → select **T4 GPU** → `Save`
2. Add your API keys to **Colab Secrets** (🔑 lock icon in left sidebar):
   - `GEMINI_API_KEY` — [get one here](https://aistudio.google.com/apikey)
   - `PEXELS_API_KEY` — [get one here](https://www.pexels.com/api/) *(optional)*
   - `GIPHY_API_KEY` — [get one here](https://developers.giphy.com/) *(optional)*
3. Run all cells top-to-bottom. Copy the **Public URL** from the last cell.

> 💡 The tunnel URL changes every Colab session.

In [ ]:
# @title 🚀 1. Setup — Clone Repo & Install Dependencies
import os, sys

REPO_URL  = "https://github.com/Sh1sh1R017/stockpile.git"
BRANCH    = "sorry-i-fucked-up"
CLONE_DIR = "/content/stockpile"

print("🔍 Checking GPU...")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo '⚠️  No GPU — switch to T4 GPU runtime!'

print("\n📦 Setting up repo...")
import os
if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print("   Already cloned — pulling latest...")
    !git -C {CLONE_DIR} fetch -q origin {BRANCH}
    !git -C {CLONE_DIR} reset -q --hard origin/{BRANCH}
else:
    !rm -rf {CLONE_DIR}
    !git clone -q -b {BRANCH} {REPO_URL} {CLONE_DIR}

%cd /content/stockpile
print("✅ Repo ready.")

In [ ]:
%%capture install_log
# @title    Install packages (output captured to keep things clean)
import subprocess, sys
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "pycloudflared", "uvicorn[standard]", "python-multipart", "nest-asyncio"], check=True)

In [ ]:
# @title    Verify install
import shutil
print("✅ Packages installed!")
for t in ["ffmpeg", "python3"]:
    p = shutil.which(t)
    print(f"  {'✅' if p else '❌'} {t}: {p or 'NOT FOUND'}")

In [ ]:
# @title 🔑 2. Configure API Keys
# RECOMMENDED: use Colab Secrets (🔑 left sidebar). Form fields below are fallback.
from pathlib import Path

GEMINI_API_KEY  = ""  # @param {type:"string"}
PEXELS_API_KEY  = ""  # @param {type:"string"}
GIPHY_API_KEY   = ""  # @param {type:"string"}
WHISPER_MODEL        = "medium"   # @param ["small", "medium", "large", "large-v3"]
WHISPER_COMPUTE_TYPE = "float32"  # @param ["float32", "float16"]

def _secret(name, fallback=""):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v.strip()
    except Exception: pass
    return fallback.strip()

gemini_key = _secret("GEMINI_API_KEY", GEMINI_API_KEY)
pexels_key = _secret("PEXELS_API_KEY", PEXELS_API_KEY)
giphy_key  = _secret("GIPHY_API_KEY",  GIPHY_API_KEY)

lines = ["LOCAL_INPUT_FOLDER=input", "LOCAL_OUTPUT_FOLDER=output",
         f"WHISPER_MODEL={WHISPER_MODEL}", f"WHISPER_COMPUTE_TYPE={WHISPER_COMPUTE_TYPE}"]
if gemini_key: lines.append(f"GEMINI_API_KEY={gemini_key}")
if pexels_key: lines.append(f"PEXELS_API_KEY={pexels_key}")
if giphy_key:  lines.append(f"GIPHY_API_KEY={giphy_key}")
Path("/content/stockpile/.env").write_text("\n".join(lines)+"\n", encoding="utf-8")

print("✅ .env written!")
print(f"   GEMINI  : {'✅' if gemini_key else '❌ missing'}")
print(f"   PEXELS  : {'✅' if pexels_key else '⚠️  not set'}")
print(f"   GIPHY   : {'✅' if giphy_key  else '⚠️  not set'}")
print(f"   Whisper : {WHISPER_MODEL} ({WHISPER_COMPUTE_TYPE})")

In [ ]:
# @title 🌐 3. Launch Backend + Cloudflare Tunnel
import os, sys, time, subprocess, urllib.request, urllib.error, nest_asyncio
from pycloudflared import try_cloudflare

nest_asyncio.apply()
os.chdir("/content/stockpile")
PORT = 8000

# Start FastAPI
_log = open("/tmp/backend.log", "w")
_proc = subprocess.Popen(
    [sys.executable, "-m", "ai_broll_autopilot.cli", "serve",
     "--host", "0.0.0.0", "--port", str(PORT)],
    stdout=_log, stderr=subprocess.STDOUT,
)

# Wait until server responds on /api/health
# NOTE: urllib.error.HTTPError = server is up (even 4xx counts as alive)
print("⏳ Waiting for backend", end="", flush=True)
server_up = False
for _ in range(30):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=2)
        server_up = True; break
    except urllib.error.HTTPError:
        server_up = True; break  # Any HTTP response = server is up
    except Exception:
        print(".", end="", flush=True)
        time.sleep(2)

if not server_up:
    print("\n❌ Backend didn't start — logs:")
    !tail -40 /tmp/backend.log
    raise RuntimeError("Backend failed to start")

print(" ✅", flush=True)

# Open Cloudflare tunnel
print("🌐 Opening Cloudflare tunnel...", flush=True)
tunnel = try_cloudflare(port=PORT)
PUBLIC_URL = tunnel.tunnel  # .tunnel is the HTTPS URL string

print("\n" + "═"*65)
print("🚀  AI B-ROLL AUTOPILOT — GPU BACKEND READY")
print("═"*65)
print(f"  🔗 Public URL  : {PUBLIC_URL}")
print(f"  🤖 Whisper     : {WHISPER_MODEL} ({WHISPER_COMPUTE_TYPE})")
print("═"*65)
print("")
print("📋 Connect your web studio:")
print(f'  PowerShell: $env:BACKEND_API_URL="{PUBLIC_URL}"; npm run dev')
print(f'  .env.local: BACKEND_API_URL={PUBLIC_URL}')
print("")
print("⚠️  URL changes every Colab session.")
print("═"*65)
print("\n🟢 Running. Stop cell (■) to shut down.\n")

hb = 0
while True:
    time.sleep(60); hb += 1
    print(f"  💓 [{hb}m] {PUBLIC_URL}", flush=True)

In [ ]:
# @title 🛠️ [Optional] View backend logs
!tail -50 /tmp/backend.log

In [ ]:
# @title 🛠️ [Optional] Health check
import urllib.request, json
try:
    with urllib.request.urlopen("http://127.0.0.1:8000/api/health") as r:
        print("✅", json.loads(r.read()))
except Exception as e:
    print("❌", e)